# 3.1 The Determinant of a Matrix

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 3 — Determinants**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.determinant import minor, cofactor, det_cofactor
from linalg_core.elimination import to_ref, swap_rows
from linalg_core import EPSILON

import numpy as np

print("Setup complete.")

---

## 2. Motivation

Throughout Chapter 2 we kept running into the same question: *is this matrix invertible?*

When the matrix was $2 \times 2$, we said it was **singular** (no inverse) when
$ad - bc = 0$ and **nonsingular** (has an inverse) otherwise. But for larger matrices
we had to perform elimination and hope no zero pivot appeared.

Is there a **single number** — one you can compute from the entries alone — that
tells you whether a matrix is invertible?

**The determinant is that number.**

$$\det(A) \neq 0 \iff A \text{ is invertible}$$

In this notebook we build the determinant from scratch:
- Start with the $2 \times 2$ formula everyone knows.
- Introduce **minors** and **cofactors** to generalize to any size.
- Show that cofactor expansion works along *any* row or column.
- Discover a fast shortcut for triangular matrices.

---

## 3. Build — Core Concepts

### 3.1 The $2 \times 2$ Determinant

> **Definition (Larson, Sec. 3.1).** The determinant of a $2 \times 2$ matrix
> $A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$ is
>
> $$\det(A) = ad - bc$$

This is the simplest case — and the base case for the recursive definition
we build below.

**Key fact:** $\det(A) = 0$ means $A$ is singular (no inverse exists).

**Example.** Consider $A = \begin{bmatrix} 3 & 1 \\ 5 & 2 \end{bmatrix}$.
Then $\det(A) = 3 \cdot 2 - 1 \cdot 5 = 1 \neq 0$, so $A$ is invertible.

Now consider $B = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}$.
Then $\det(B) = 1 \cdot 4 - 2 \cdot 2 = 0$, so $B$ is singular.
Note that row 2 is exactly $2 \times$ row 1 — the rows are linearly dependent.

In [ ]:
A = Matrix.from_lists([[3, 1], [5, 2]])
B = Matrix.from_lists([[1, 2], [2, 4]])

det_A = A[0,0] * A[1,1] - A[0,1] * A[1,0]
det_B = B[0,0] * B[1,1] - B[0,1] * B[1,0]

print("A ="); print(A)
print(f"\ndet(A) = ({A[0,0]:.0f})({A[1,1]:.0f}) - ({A[0,1]:.0f})({A[1,0]:.0f}) = {det_A:.0f}")
print(f"A is invertible: {abs(det_A) > EPSILON}")

print("\nB ="); print(B)
print(f"\ndet(B) = ({B[0,0]:.0f})({B[1,1]:.0f}) - ({B[0,1]:.0f})({B[1,0]:.0f}) = {det_B:.0f}")
print(f"B is singular: {abs(det_B) < EPSILON}")

print("\nVerify with det_cofactor:")
print(f"det_cofactor(A) = {det_cofactor(A):.0f}")
print(f"det_cofactor(B) = {det_cofactor(B):.0f}")

### 3.2 Minors — Deleting a Row and a Column

> **Definition (Larson, Sec. 3.1).** The **minor** $M_{ij}$ of a square matrix $A$
> is the determinant of the submatrix formed by deleting row $i$ and column $j$.

For a $3 \times 3$ matrix, deleting one row and one column leaves a $2 \times 2$ matrix,
so each minor is just a $2 \times 2$ determinant.

**Example.** Let
$$A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 0 \end{bmatrix}$$

The minor $M_{00}$ is the determinant of $A$ with row 0 and column 0 deleted:

$$M_{00} = \det\begin{bmatrix} 5 & 6 \\ 8 & 0 \end{bmatrix} = 5 \cdot 0 - 6 \cdot 8 = -48$$

Our `minor(A, i, j)` function does this automatically.

In [ ]:
A = Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 0]])
print("A ="); print(A)

print("\n--- Minors along row 0 ---")
for j in range(3):
    m = minor(A, 0, j)
    print(f"M_0{j} = minor(A, 0, {j}) = {m:.0f}")

print("\n--- Spot check: M_00 by hand ---")
print("Delete row 0, col 0 -> submatrix [[5,6],[8,0]]")
print(f"det = 5*0 - 6*8 = {5*0 - 6*8}")
print(f"minor(A, 0, 0) = {minor(A, 0, 0):.0f}  (matches)")

### 3.3 Cofactors — Adding the Sign

> **Definition (Larson, Sec. 3.1).** The **cofactor** $C_{ij}$ is the signed minor:
>
> $$C_{ij} = (-1)^{i+j} \, M_{ij}$$

The sign follows a **checkerboard pattern**:

$$\begin{bmatrix} + & - & + & \cdots \\ - & + & - & \cdots \\ + & - & + & \cdots \\ \vdots & \vdots & \vdots & \ddots \end{bmatrix}$$

Position $(0,0)$ gets $+$, $(0,1)$ gets $-$, and so on.

This sign correction is what makes the cofactor expansion formula work.

In [ ]:
A = Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 0]])
print("A ="); print(A)

print("\n--- Checkerboard sign pattern (3x3) ---")
for i in range(3):
    signs = []
    for j in range(3):
        sign = (-1) ** (i + j)
        signs.append(f"{'+' if sign > 0 else '-':>2}")
    print("  " + "  ".join(signs))

print("\n--- Cofactors along row 0 ---")
for j in range(3):
    m = minor(A, 0, j)
    sign = (-1) ** (0 + j)
    c = cofactor(A, 0, j)
    print(f"C_0{j} = ({'+1' if sign > 0 else '-1'}) * M_0{j} = ({'+1' if sign > 0 else '-1'}) * ({m:.0f}) = {c:.0f}")

### 3.4 Cofactor Expansion Along Row 0

> **Theorem (Larson, Sec. 3.1).** The determinant of an $n \times n$ matrix $A$
> can be computed by expanding along row 0:
>
> $$\det(A) = \sum_{j=0}^{n-1} a_{0j} \, C_{0j}$$
>
> where $C_{0j} = (-1)^{0+j} M_{0j}$ is the $(0, j)$ cofactor.

This is a **recursive** definition: the determinant of an $n \times n$ matrix
depends on determinants of $(n-1) \times (n-1)$ matrices, which in turn depend
on determinants of $(n-2) \times (n-2)$ matrices, all the way down to the
$1 \times 1$ (or $2 \times 2$) base case.

Our `det_cofactor(A)` implements exactly this recursion.

In [ ]:
A = Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 0]])
print("A ="); print(A)

print("\n--- Cofactor expansion along row 0 ---")
print("det(A) = a_00 * C_00  +  a_01 * C_01  +  a_02 * C_02")

total = 0.0
for j in range(3):
    a_0j = A[0, j]
    c_0j = cofactor(A, 0, j)
    term = a_0j * c_0j
    total += term
    print(f"       = ({a_0j:.0f}) * ({c_0j:.0f})  =  {term:.0f}")

print(f"\ndet(A) = {total:.0f}")
print(f"det_cofactor(A) = {det_cofactor(A):.0f}")

### 3.5 Detailed 3×3 Demo — Step by Step

Let us walk through every cofactor for

$$A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 0 \end{bmatrix}$$

expanding along row 0:

$$\det(A) = 1 \cdot C_{00} + 2 \cdot C_{01} + 3 \cdot C_{02}$$

**Cofactor $C_{00}$:** sign $= (+1)$, submatrix $= \begin{bmatrix} 5 & 6 \\ 8 & 0 \end{bmatrix}$, minor $= 0 - 48 = -48$, cofactor $= -48$.

**Cofactor $C_{01}$:** sign $= (-1)$, submatrix $= \begin{bmatrix} 4 & 6 \\ 7 & 0 \end{bmatrix}$, minor $= 0 - 42 = -42$, cofactor $= +42$.

**Cofactor $C_{02}$:** sign $= (+1)$, submatrix $= \begin{bmatrix} 4 & 5 \\ 7 & 8 \end{bmatrix}$, minor $= 32 - 35 = -3$, cofactor $= -3$.

$$\det(A) = 1(-48) + 2(42) + 3(-3) = -48 + 84 - 9 = 27$$

In [ ]:
A = Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 0]])
print("A ="); print(A)
print()

print("=" * 55)
print("Step-by-step cofactor expansion along row 0")
print("=" * 55)

running_total = 0.0
for j in range(3):
    sign = (-1) ** (0 + j)
    sign_str = "+1" if sign > 0 else "-1"
    m = minor(A, 0, j)
    c = cofactor(A, 0, j)
    a_0j = A[0, j]
    term = a_0j * c
    running_total += term

    sub_rows = []
    for r in range(3):
        if r == 0:
            continue
        row = []
        for k in range(3):
            if k == j:
                continue
            row.append(f"{A[r,k]:.0f}")
        sub_rows.append("[" + ", ".join(row) + "]")
    sub_str = "[" + ", ".join(sub_rows) + "]"

    print(f"\nC_0{j}:")
    print(f"  Sign: (-1)^(0+{j}) = {sign_str}")
    print(f"  Submatrix (delete row 0, col {j}): {sub_str}")
    print(f"  Minor M_0{j} = {m:.0f}")
    print(f"  Cofactor C_0{j} = ({sign_str}) * ({m:.0f}) = {c:.0f}")
    print(f"  Term: a_0{j} * C_0{j} = ({a_0j:.0f}) * ({c:.0f}) = {term:.0f}")

print("\n" + "=" * 55)
print(f"det(A) = {running_total:.0f}")
print(f"det_cofactor(A) = {det_cofactor(A):.0f}  (matches)")

### 3.6 Expansion Along Any Row or Column

> **Theorem (Larson, Sec. 3.1).** The cofactor expansion can be performed along
> **any** row or **any** column, and the result is the same:
>
> **Row $i$:** $\displaystyle \det(A) = \sum_{j=0}^{n-1} a_{ij} \, C_{ij}$
>
> **Column $j$:** $\displaystyle \det(A) = \sum_{i=0}^{n-1} a_{ij} \, C_{ij}$

This is a powerful result: you can **choose** the row or column with the most
zeros to minimize the computation.

Let us verify by computing $\det(A)$ along row 0, row 1, column 0, and column 2
for the same matrix.

In [ ]:
A = Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 0]])
print("A ="); print(A)


def det_expand_row(A, i):
    """Cofactor expansion along row i."""
    return sum(A[i, j] * cofactor(A, i, j) for j in range(A.cols))


def det_expand_col(A, j):
    """Cofactor expansion along column j."""
    return sum(A[i, j] * cofactor(A, i, j) for i in range(A.rows))


print("\n--- Expansion along different rows and columns ---")
d_row0 = det_expand_row(A, 0)
d_row1 = det_expand_row(A, 1)
d_col0 = det_expand_col(A, 0)
d_col2 = det_expand_col(A, 2)

print(f"Row 0:    det = {d_row0:.0f}")
print(f"Row 1:    det = {d_row1:.0f}")
print(f"Column 0: det = {d_col0:.0f}")
print(f"Column 2: det = {d_col2:.0f}")

all_same = (
    abs(d_row0 - d_row1) < EPSILON
    and abs(d_row0 - d_col0) < EPSILON
    and abs(d_row0 - d_col2) < EPSILON
)
print(f"\nAll four give the same result: {all_same}")
print("Expansion along any row or column yields the same determinant.")

### 3.7 Triangular Matrix Shortcut

> **Theorem (Larson, Sec. 3.1).** If $A$ is a **triangular** matrix (upper or lower),
> then its determinant is the **product of its diagonal entries**:
>
> $$\det(A) = a_{00} \cdot a_{11} \cdots a_{n-1,\,n-1}$$

This follows from cofactor expansion: in an upper triangular matrix, expanding
along column 0 gives only one nonzero term ($a_{00}$), which multiplies the
cofactor — itself an upper triangular determinant. The recursion peels off
one diagonal element at a time.

**Practical shortcut:** Reduce $A$ to row-echelon form (upper triangular) using
elimination, then multiply the diagonal. You must track row swaps (each swap
flips the sign) and remember that our `to_ref` scales pivot rows to have
leading 1s, so we need to account for the original pivot values.

Below we demonstrate the principle directly: reduce a copy to REF, then show
that $\det = (\text{sign}) \times \text{product of original pivots}$.

In [ ]:
print("--- Example 1: Upper triangular matrix ---")
U = Matrix.from_lists([[2, 3, 1], [0, -4, 5], [0, 0, 3]])
print("U ="); print(U)

diag_product = U[0,0] * U[1,1] * U[2,2]
print(f"\nProduct of diagonal: ({U[0,0]:.0f}) * ({U[1,1]:.0f}) * ({U[2,2]:.0f}) = {diag_product:.0f}")
print(f"det_cofactor(U) = {det_cofactor(U):.0f}")
print(f"Match: {abs(diag_product - det_cofactor(U)) < EPSILON}")

print("\n--- Example 2: Lower triangular matrix ---")
L = Matrix.from_lists([[5, 0, 0], [2, -3, 0], [1, 4, 7]])
print("L ="); print(L)

diag_product_L = L[0,0] * L[1,1] * L[2,2]
print(f"\nProduct of diagonal: ({L[0,0]:.0f}) * ({L[1,1]:.0f}) * ({L[2,2]:.0f}) = {diag_product_L:.0f}")
print(f"det_cofactor(L) = {det_cofactor(L):.0f}")
print(f"Match: {abs(diag_product_L - det_cofactor(L)) < EPSILON}")

In [ ]:
print("--- Example 3: General matrix -> REF -> diagonal product ---")
A = Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 0]])
print("A ="); print(A)

M = A.copy()
n = M.rows
sign = 1
pivot_vals = []

for col in range(n):
    best = col
    for r in range(col + 1, n):
        if abs(M[r, col]) > abs(M[best, col]):
            best = r
    if abs(M[best, col]) < EPSILON:
        print(f"Zero pivot at column {col} -> det = 0")
        break
    if best != col:
        swap_rows(M, col, best)
        sign *= -1
    pivot_vals.append(M[col, col])
    for r in range(col + 1, n):
        if abs(M[r, col]) > EPSILON:
            factor = M[r, col] / M[col, col]
            for k in range(col, n):
                M[r, k] -= factor * M[col, k]

print("\nAfter elimination (upper triangular):")
print(M)

diag_prod = 1.0
for i in range(n):
    diag_prod *= M[i, i]

det_from_ref = sign * diag_prod
print(f"\nRow swaps: sign = {'+' if sign > 0 else '-'}1")
print(f"Diagonal product: {diag_prod:.1f}")
print(f"det = sign * diag_product = ({sign}) * ({diag_prod:.1f}) = {det_from_ref:.1f}")
print(f"det_cofactor(A) = {det_cofactor(A):.1f}")
print(f"Match: {abs(det_from_ref - det_cofactor(A)) < EPSILON}")

---

## 4. Verify — Cross-Check with NumPy

Our from-scratch `det_cofactor` should agree with `np.linalg.det` for every matrix.
We also verify the triangular shortcut and confirm that the $2 \times 2$ formula
matches the general cofactor expansion.

In [ ]:
def to_np(mat):
    """Convert our Matrix to a NumPy array."""
    return np.array(mat.to_lists())


def check_det(mat, label):
    """Compare det_cofactor vs np.linalg.det."""
    ours = det_cofactor(mat)
    theirs = np.linalg.det(to_np(mat))
    match = abs(ours - theirs) < 1e-6
    status = "PASS" if match else "FAIL"
    print(f"[{status}] {label}: ours={ours:.6f}, numpy={theirs:.6f}")
    return match

In [ ]:
print("=" * 60)
print("VERIFICATION: det_cofactor vs np.linalg.det")
print("=" * 60)

test_matrices = [
    (Matrix.from_lists([[3, 1], [5, 2]]), "2x2 invertible"),
    (Matrix.from_lists([[1, 2], [2, 4]]), "2x2 singular"),
    (Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 0]]), "3x3 #1"),
    (Matrix.from_lists([[2, -1, 0], [3, 1, 4], [-2, 5, 3]]), "3x3 #2"),
    (Matrix.from_lists([[1, 0, 0], [0, 1, 0], [0, 0, 1]]), "3x3 identity"),
    (Matrix.from_lists([
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [2, 1, 4, 3],
        [6, 8, 5, 7]
    ]), "4x4 general"),
    (Matrix.from_lists([
        [1, 0, 0, 0],
        [0, 2, 0, 0],
        [0, 0, 3, 0],
        [0, 0, 0, 4]
    ]), "4x4 diagonal"),
]

all_pass = True
for mat, label in test_matrices:
    if not check_det(mat, label):
        all_pass = False

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION: Triangular = diagonal product")
print("=" * 60)

tri_tests = [
    Matrix.from_lists([[2, 3, 1], [0, -4, 5], [0, 0, 3]]),
    Matrix.from_lists([[5, 0, 0], [2, -3, 0], [1, 4, 7]]),
    Matrix.from_lists([[1, 0, 0, 0], [0, 2, 0, 0], [0, 0, 3, 0], [0, 0, 0, 4]]),
    Matrix.from_lists([[3, 1, 4, 1], [0, 5, 9, 2], [0, 0, 6, 5], [0, 0, 0, 3]]),
]

all_pass = True
for idx, T in enumerate(tri_tests):
    diag_prod = 1.0
    for i in range(T.rows):
        diag_prod *= T[i, i]
    det_val = det_cofactor(T)
    match = abs(diag_prod - det_val) < EPSILON
    status = "PASS" if match else "FAIL"
    print(f"[{status}] Triangular #{idx+1}: diag_product={diag_prod:.2f}, det_cofactor={det_val:.2f}")
    if not match:
        all_pass = False

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION: 2x2 formula matches cofactor expansion")
print("=" * 60)

import random
random.seed(42)

all_pass = True
for trial in range(10):
    data = [random.uniform(-10, 10) for _ in range(4)]
    M = Matrix(2, 2, data)
    formula = M[0,0] * M[1,1] - M[0,1] * M[1,0]
    cofactor_val = det_cofactor(M)
    match = abs(formula - cofactor_val) < EPSILON
    status = "PASS" if match else "FAIL"
    print(f"[{status}] Trial {trial+1}: ad-bc={formula:.4f}, cofactor={cofactor_val:.4f}")
    if not match:
        all_pass = False

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

---

## 5. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Compute the determinant of

$$A = \begin{bmatrix} 3 & -2 \\ 6 & -4 \end{bmatrix}$$

using the $2 \times 2$ formula $ad - bc$. Then verify with `det_cofactor(A)`.

Is this matrix singular or invertible? Why does the result make geometric sense
(think about the relationship between the two rows)?

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Compute the determinant of

$$B = \begin{bmatrix} 2 & -1 & 3 \\ 1 & 0 & -2 \\ 4 & 1 & 5 \end{bmatrix}$$

in **two different ways**:

1. By cofactor expansion along **row 1** (the row with a zero — fewer terms!).
2. By cofactor expansion along **column 2**.

Show each term explicitly and verify both give the same answer.
Then check with `det_cofactor(B)`.

*Hint:* Expanding along a row or column with zeros saves work because
$a_{ij} \cdot C_{ij} = 0$ whenever $a_{ij} = 0$.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

The cofactor expansion algorithm has factorial time complexity: $O(n!)$ for an
$n \times n$ matrix. This means it becomes **extremely slow** for large matrices.

Write a timing experiment:
1. Generate random $n \times n$ matrices for $n = 2, 3, 4, 5, 6, 7, 8, 9, 10$.
2. Time `det_cofactor` for each $n$ (use `time.time()` or `time.perf_counter()`).
3. Plot the results on a log-scale y-axis.

You should see roughly exponential growth. Compare visually against $n!$ to
confirm the factorial scaling.

*Note:* For $n \geq 11$, `det_cofactor` may take several minutes. In Notebook 3.2
we will build an $O(n^3)$ algorithm using elimination.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **2x2 determinant** | $\det\begin{bmatrix} a & b \\ c & d \end{bmatrix} = ad - bc$ |
| **Minor $M_{ij}$** | Determinant of the submatrix with row $i$, col $j$ deleted |
| **Cofactor $C_{ij}$** | $(-1)^{i+j} M_{ij}$ — the signed minor |
| **Cofactor expansion** | $\det(A) = \sum_j a_{ij} C_{ij}$ along any row $i$ (or column) |
| **Any row/column works** | The expansion gives the same result regardless of which row or column you choose |
| **Triangular shortcut** | $\det = $ product of diagonal entries |
| **Singularity test** | $\det(A) = 0 \iff A$ is singular (not invertible) |
| **Complexity** | Cofactor expansion is $O(n!)$ — fast for small $n$, impractical for large $n$ |

**Next:** In Notebook 3.2 we use elimination to compute determinants in $O(n^3)$ time
and explore how row operations affect the determinant.